# Coordinate and time distributions

Run from this notebook directory in the matching PycWB environment. This exercise generates parameter locations only; it does not run a search or evaluate waveforms. Angles shown on these plots are radians.


In [ ]:
# add current directory to path
import sys
sys.path.insert(0, '..')

import pycwb

print("Version used for this test: %s" % pycwb.__version__)

In [ ]:
from pycwb.config import Config
from pycwb.modules.logger import logger_init
from pycwb.modules.injection.par_generator import get_injection_list_from_parameters, repeat
from pycwb.modules.injection.sky_distribution import generate_sky_distribution, distribute_injections_on_sky
from pycwb.modules.injection.injection import distribute_inj_in_gps_time_by_rate
from math import ceil
import numpy as np

logger_init()

injection = {
    'rate': '1/200',
    'jitter': 50,
    'allow_reuse_data': True,
    'repeat_injection': 10,
    'sky_distribution': {
        'type': 'Fixed',
        'coordsys': 'geo',
        'coordinates': {'longitude': '-110 deg', 'latitude': '50 deg'}
    },
    'parameters_from_python': {
        'function': 'injection_parameters.py.get_injection_parameters',
    },
}


repeat_injection = injection['repeat_injection']
rate = eval(injection['rate'])
jitter = injection['jitter']
sky_distribution = injection['sky_distribution']

print(f"Repeat injection: {repeat_injection}")
print(f"Rate: {rate}")
print(f"Jitter: {jitter}")
print(f"Sky distribution: {sky_distribution}")

total_live_time = 3600 * 3 
start_gps_time = 1262304000
end_gps_time = start_gps_time + total_live_time
injections = get_injection_list_from_parameters(injection)
print(f"Number of injections: {len(injections)}")
injections = repeat(injections, repeat_injection)
print(f"Number of injections after repeat: {len(injections)}")


%matplotlib inline
import matplotlib.pyplot as plt

distribute_inj_in_gps_time_by_rate(injections, rate, jitter, start_gps_time, end_gps_time, shuffle=False)

plt.figure(figsize=(4, 4))
plt.plot([inj['gps_time'] for inj in injections], [inj['trial_idx'] for inj in injections], 'o')
plt.xlim(start_gps_time, end_gps_time)
plt.show()

## Patch sky distribution

In [ ]:
%matplotlib inline
from pycwb.utils.skymap_coord import convert_to_celestial_coordinates
import matplotlib.pyplot as plt

sky_distribution = {
    'type': 'Patch',
    'coordsys': 'geo', 
    'patch': {
        'center': {'longitude': '45 deg', 'latitude': '45 deg'},
        'radius': '10 deg'
    }
}

sky_locations = generate_sky_distribution(sky_distribution, len(injections))
print(f"Number of sky locations: {len(sky_locations[0])}")
distribute_injections_on_sky(injections, sky_locations, coordsys=sky_distribution['coordsys'])
print(f"Number of injections after sky distribution: {len(injections)}")

phi = np.array([inj['sky_loc'][0] for inj in injections])
theta = np.array([inj['sky_loc'][1] for inj in injections])

celestial_coordinates = [convert_to_celestial_coordinates(inj['sky_loc'][0], inj['sky_loc'][1], inj['gps_time'], inj['coordsys']) for inj in injections] 

ras = np.array([coord[0] for coord in celestial_coordinates])
decs = np.array([coord[1] for coord in celestial_coordinates]) 

# plot sky distribution
fig, ax = plt.subplots(1,2, figsize=(8, 4))
ax[0].scatter(phi, theta, s=1)
ax[1].scatter(ras, decs, s=1) 
ax[0].set_title('Original Coordinates')
ax[0].set_xlabel('Longitude (rad)')
ax[0].set_ylabel('Latitude (rad)')
ax[1].set_title('Celestial Coordinates')
ax[1].set_xlabel('RA')
ax[1].set_ylabel('Dec')
ax[0].set_xlim(0,2*np.pi)
ax[1].set_xlim(0,2*np.pi)
plt.show()


## Distribute at a Fixed Point

In [ ]:

%matplotlib inline
from pycwb.utils.skymap_coord import convert_to_celestial_coordinates
import matplotlib.pyplot as plt

sky_distribution = {
    'type': 'Fixed',
    'coordsys': 'geo',
    'coordinates': {
        'longitude': '60 deg', 'latitude': '10 deg' 
    }
}


sky_locations = generate_sky_distribution(sky_distribution, len(injections))
print(f"Number of sky locations: {len(sky_locations[0])}")
distribute_injections_on_sky(injections, sky_locations, coordsys=sky_distribution['coordsys'])
print(f"Number of injections after sky distribution: {len(injections)}")

phi = np.array([inj['sky_loc'][0] for inj in injections])
theta = np.array([inj['sky_loc'][1] for inj in injections])

celestial_coordinates = [convert_to_celestial_coordinates(inj['sky_loc'][0], inj['sky_loc'][1], inj['gps_time'], inj['coordsys']) for inj in injections] 

ras = np.array([coord[0] for coord in celestial_coordinates])
decs = np.array([coord[1] for coord in celestial_coordinates]) 

# plot sky distribution
fig, ax = plt.subplots(1,2, figsize=(8, 4))
ax[0].scatter(phi, theta, s=1)
ax[1].scatter(ras, decs, s=1) 
ax[0].set_title('Original Coordinates')
ax[0].set_xlabel('Longitude (rad)')
ax[0].set_ylabel('Latitude (rad)')
ax[1].set_title('Celestial Coordinates')
ax[1].set_xlabel('RA')
ax[1].set_ylabel('Dec')
ax[1].set_xlim(0,2*np.pi)
plt.show()
